# Project 1: Data Cleaning & Preparation
**Company:** DecodeLabs | **Batch:** 2026

## Goal
Clean a raw e-commerce orders dataset by handling missing values, duplicates, incorrect formats, and data-integrity issues — producing a production-ready, analysis-safe dataset.

In [1]:

import pandas as pd
import numpy as np
import os

os.makedirs('charts', exist_ok=True)

# ---- Load the raw dataset provided by DecodeLabs ----
df_raw = pd.read_excel('raw_dataset.xlsx')
print("Shape:", df_raw.shape)
df_raw.head(10)


Shape: (1200, 14)


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
5,ORD200005,2023-10-23,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72
6,ORD200006,2025-06-17,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42
7,ORD200007,2023-05-12,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,NaN,Email,268.56
9,ORD200009,2023-11-21,C31946,Desk,4,509.38,102 Main St,Credit Card,Shipped,TRK33478363,6,SAVE10,Google,2037.52


## Phase 0 — Data Quality Report (Before Cleaning)

In [2]:

before_report = {
    'rows': len(df_raw),
    'columns': df_raw.shape[1],
    'duplicate_rows': df_raw.duplicated().sum(),
    'duplicate_order_ids': df_raw['OrderID'].duplicated().sum(),
    'duplicate_tracking_numbers': df_raw['TrackingNumber'].duplicated().sum(),
    'null_counts': df_raw.isnull().sum().to_dict(),
}
for k, v in before_report.items():
    print(k, ":", v)


rows : 1200
columns : 14
duplicate_rows : 0
duplicate_order_ids : 0
duplicate_tracking_numbers : 0
null_counts : {'OrderID': 0, 'Date': 0, 'CustomerID': 0, 'Product': 0, 'Quantity': 0, 'UnitPrice': 0, 'ShippingAddress': 0, 'PaymentMethod': 0, 'OrderStatus': 0, 'TrackingNumber': 0, 'ItemsInCart': 0, 'CouponCode': 309, 'ReferralSource': 0, 'TotalPrice': 0}


**Findings:** `CouponCode` has 309 missing values (~26%) — customers who did not use a coupon. `OrderID` and `TrackingNumber` are confirmed unique (0 duplicates), which is a good sign for referential integrity.

## Phase 1 — Strategic Imputation (Handle the Gaps, Don't Just Delete)

**Decision:** `CouponCode` missing = customer simply did not apply a coupon, **not** a data-entry error. Deleting these rows would discard 26% of valid orders. We impute with the explicit label `'NoCoupon'` rather than mean/median/mode, since this is a categorical *absence* signal, not a true missing value.

In [3]:

df = df_raw.copy()
df['CouponCode'] = df['CouponCode'].fillna('NoCoupon')
print("Nulls remaining:\n", df.isnull().sum())


Nulls remaining:
 OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64


## Phase 2 — The Integrity Audit (One Truth, One Record)

In [4]:

# Exact duplicate rows
dup_rows = df.duplicated().sum()
df = df.drop_duplicates()

# Duplicate unique identifiers (OrderID must be truly unique)
dup_orderid = df['OrderID'].duplicated().sum()
dup_tracking = df['TrackingNumber'].duplicated().sum()

print(f"Exact duplicate rows removed: {dup_rows}")
print(f"Duplicate OrderIDs remaining: {dup_orderid}")
print(f"Duplicate TrackingNumbers remaining: {dup_tracking}")
assert dup_orderid == 0, "OrderID must be 100% unique!"
assert dup_tracking == 0, "TrackingNumber must be 100% unique!"
print("\nIntegrity check PASSED: OrderID and TrackingNumber are 100% unique.")


Exact duplicate rows removed: 0
Duplicate OrderIDs remaining: 0
Duplicate TrackingNumbers remaining: 0

Integrity check PASSED: OrderID and TrackingNumber are 100% unique.


## Phase 3 — Speak One Language (Standardise Formatting)

In [5]:

# Trim whitespace & standardise case for all text/categorical columns
text_cols = ['Product','ShippingAddress','PaymentMethod','OrderStatus','CouponCode','ReferralSource']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Title-case categorical fields for consistent display
for col in ['Product','PaymentMethod','OrderStatus','ReferralSource']:
    df[col] = df[col].str.title()

# ISO 8601 date format (YYYY-MM-DD) — verify & enforce
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
invalid_dates = df['Date'].isnull().sum()
print(f"Rows with unparseable dates: {invalid_dates}")
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# Numeric precision -> 2 decimals for currency fields
df['UnitPrice'] = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)

df.head(10)


Rows with unparseable dates: 0


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
5,ORD200005,2023-10-23,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72
6,ORD200006,2025-06-17,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42
7,ORD200007,2023-05-12,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,NoCoupon,Email,268.56
9,ORD200009,2023-11-21,C31946,Desk,4,509.38,102 Main St,Credit Card,Shipped,TRK33478363,6,SAVE10,Google,2037.52


**Integrity check:** 0% error rate on date formats confirmed — every `Date` value now conforms to ISO 8601 (`YYYY-MM-DD`).

## Phase 4 — Business Logic Validation (TotalPrice Recalculation)

In [6]:

# Cross-check TotalPrice = Quantity * UnitPrice
expected_total = (df['Quantity'] * df['UnitPrice']).round(2)
mismatch_mask = (expected_total != df['TotalPrice'])
mismatch_count = mismatch_mask.sum()
print(f"Rows where TotalPrice did not match Quantity x UnitPrice: {mismatch_count}")

# Decision: recalculate TotalPrice from source fields (Quantity, UnitPrice) since these
# are the primary transactional inputs and less likely to have been manually mis-typed
# than a derived/computed field.
df.loc[mismatch_mask, 'TotalPrice'] = expected_total[mismatch_mask]
print(f"Corrected {mismatch_count} TotalPrice values to match Quantity x UnitPrice.")


Rows where TotalPrice did not match Quantity x UnitPrice: 0
Corrected 0 TotalPrice values to match Quantity x UnitPrice.


## Phase 5 — Outlier Detection (IQR Method on TotalPrice)

In [7]:

Q1 = df['TotalPrice'].quantile(0.25)
Q3 = df['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = df[(df['TotalPrice'] < lower) | (df['TotalPrice'] > upper)]
print(f"TotalPrice outliers detected: {len(outliers)} (bounds: [{lower:.2f}, {upper:.2f}])")
print("Decision: retained as valid (large bulk orders are plausible for this business), flagged for review only.")
df['IsPriceOutlier'] = ((df['TotalPrice'] < lower) | (df['TotalPrice'] > upper)).astype(int)


TotalPrice outliers detected: 8 (bounds: [-1341.41, 3330.41])
Decision: retained as valid (large bulk orders are plausible for this business), flagged for review only.


## Phase 6 — Data Type Correction

In [8]:

df['Date'] = pd.to_datetime(df['Date'])
df['OrderID'] = df['OrderID'].astype(str)
df['CustomerID'] = df['CustomerID'].astype(str)
df['PaymentMethod'] = df['PaymentMethod'].astype('category')
df['OrderStatus'] = df['OrderStatus'].astype('category')
print(df.dtypes)


OrderID                       str
Date               datetime64[us]
CustomerID                    str
Product                       str
Quantity                    int64
UnitPrice                 float64
ShippingAddress               str
PaymentMethod            category
OrderStatus              category
TrackingNumber                str
ItemsInCart                 int64
CouponCode                    str
ReferralSource                str
TotalPrice                float64
IsPriceOutlier              int64
dtype: object


## Phase 7 — Before vs After Summary

In [9]:

after_report = {
    'rows': len(df),
    'duplicate_rows': df.duplicated().sum(),
    'duplicate_order_ids': df['OrderID'].duplicated().sum(),
    'total_nulls': int(df.isnull().sum().sum()),
}
summary = pd.DataFrame({
    'Metric': ['Row Count','Duplicate Rows','Duplicate OrderIDs','Total Nulls','CouponCode Nulls'],
    'Before': [before_report['rows'], before_report['duplicate_rows'], before_report['duplicate_order_ids'],
               sum(before_report['null_counts'].values()), before_report['null_counts']['CouponCode']],
    'After':  [after_report['rows'], after_report['duplicate_rows'], after_report['duplicate_order_ids'],
               after_report['total_nulls'], 0]
})
summary


,Metric,Before,After
0,Row Count,1200,1200
1,Duplicate Rows,0,0
2,Duplicate OrderIDs,0,0
3,Total Nulls,309,0
4,CouponCode Nulls,309,0


## Phase 8 — Save Cleaned, Production-Ready Dataset

In [10]:

df.to_excel('cleaned_dataset.xlsx', index=False)
df.to_csv('cleaned_dataset.csv', index=False)
print("Cleaned dataset saved as cleaned_dataset.xlsx / .csv")
df.head(10)


Cleaned dataset saved as cleaned_dataset.xlsx / .csv


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,IsPriceOutlier
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10,0
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70,0
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40,0
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19,0
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04,0
5,ORD200005,2023-10-23,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72,0
6,ORD200006,2025-06-17,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42,0
7,ORD200007,2023-05-12,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75,0
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,NoCoupon,Email,268.56,0
9,ORD200009,2023-11-21,C31946,Desk,4,509.38,102 Main St,Credit Card,Shipped,TRK33478363,6,SAVE10,Google,2037.52,0


## Conclusion

This dataset moved from **raw operational export** to **"Gold Standard" analysis-ready data**:
- ✅ 0% duplicate unique identifiers (OrderID, TrackingNumber)
- ✅ 0% incorrectly formatted dates (all ISO 8601)
- ✅ 0 remaining nulls (CouponCode gap explicitly labelled, not guessed)
- ✅ Business-logic integrity restored (TotalPrice = Quantity × UnitPrice for all rows)
- ✅ Every change documented in the accompanying **Change Log** (see `Change_Log.pdf`)

This is the foundation phase — Project 2 (analysis/modeling) can now be built on trustworthy data.